**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *DEMONSTRATE A LOST-UPDATE RACE CONDITION AND PROTECT SHARED STATE WITH A LOCK. 🐍🔒***

Two workers each add one to a shared counter. Each operation works correctly on its own—but together they can overwrite one another's work.

A **race condition** occurs when correctness depends on the timing or ordering of concurrent operations. Here, the vulnerable operation is a **read-modify-write** sequence.

In this Python Systems Engineering example, we deliberately arrange an unsafe interleaving with a barrier, inspect the wrong result, and use a shared lock to protect the complete update. This teaches coordination and correctness, not performance.

---

### Requirements

Python 3.9+ and a Jupyter environment. All example imports are from the Python standard library, so **no additional package installation is needed**. Run the cells from top to bottom. The barriers use a timeout so a missing participant raises an error rather than waiting indefinitely.

---

### 1. Import the concurrency tools

`ThreadPoolExecutor` runs two workers and surfaces worker exceptions through futures. `Barrier` coordinates their progress; `Lock` allows only one cooperating worker into the protected update at a time.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier, Lock


---

### 2. Reproduce the lost update

The expected final counter is **2**. Each worker reads **0**, then waits at the barrier. Once both have read, both write **1**. One increment is lost.

The barrier is **test scaffolding**: it deliberately selects a problematic interleaving. It does not protect the counter, and the outcome is not a measurement of how often an unconstrained program races. No `sleep()` or repeated lucky scheduling is needed.

In [2]:
def lost_update_demo():
    counter = 0
    both_read = Barrier(2, timeout=5)

    def increment():
        nonlocal counter
        snapshot = counter              # Read shared state.
        both_read.wait()                 # Force BOTH reads before either write.
        counter = snapshot + 1           # Both workers write 1.
        return snapshot

    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(increment) for _ in range(2)]
        reads = [future.result() for future in futures]
    return counter, reads

unsafe_result, reads = lost_update_demo()
print("Values read:", reads)
print("Expected: 2 | Actual:", unsafe_result)
assert reads == [0, 0] and unsafe_result == 1


Values read: [0, 0]
Expected: 2 | Actual: 1


---

### 3. Protect the critical section

Use **one shared lock** around the read and the write. The first worker changes 0 to 1; the next reads 1 and changes it to 2. Which worker goes first is unspecified.

The start barrier is outside the lock. Do **not** put a two-worker barrier inside this critical section: the first worker would wait while preventing the second from entering. Locking only the final assignment would also leave the earlier read exposed.

In [3]:
def locked_demo():
    counter = 0
    lock = Lock()                        # One shared lock for both workers.
    ready = Barrier(2, timeout=5)

    def increment():
        nonlocal counter
        ready.wait()                     # Rendezvous BEFORE acquiring the lock.
        with lock:                       # Protect the ENTIRE read-modify-write.
            snapshot = counter
            counter = snapshot + 1
        return snapshot

    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(increment) for _ in range(2)]
        reads = [future.result() for future in futures]
    return counter, reads

safe_result, reads = locked_demo()
print("Values read (sorted):", sorted(reads))
print("Expected: 2 | Actual:", safe_result)
assert sorted(reads) == [0, 1] and safe_result == 2


Values read (sorted): [0, 1]
Expected: 2 | Actual: 2


---

### 4. Verify both controlled cases

Repeat with fresh state. These assertions check this deliberately coordinated example; they are not a general proof of thread safety or a race-frequency benchmark.

In [4]:
# Fresh state and synchronization objects for every run.
trials = 20
for _ in range(trials):
    assert lost_update_demo()[0] == 1
    assert locked_demo()[0] == 2
print(f"{trials} controlled runs: unsafe = 1, locked = 2")


20 controlled runs: unsafe = 1, locked = 2


---

### Takeaways and limits

- Individually correct operations do not automatically compose into a correct concurrent operation.
- Protect the entire read-modify-write sequence, with every competing accessor following the same synchronization policy.
- This lock coordinates threads in **one process**. Database or cross-process updates require synchronization appropriate to that shared resource.
- The GIL, where enabled, does not make a multi-step application operation a transaction. This example explicitly separates the read from the write and yields at a barrier; it makes no claim about the scheduling of a bare `counter += 1` on every Python build.
- Locks serialize the protected work. Keep critical sections focused and consider contention or deadlocks when applying them to larger systems.

---